In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. Configuration & Directories
SEED = 42
BATCH_SIZE = 64        # 64 balances GPU memory usage and gradient stability for 63k images
IMAGE_SIZE = (224, 224)
NUM_WORKERS = 4        # Set to 0 if encountering multiprocessing issues on Windows

TRAIN_CSV = "train_labels.csv"
SAMPLE_SUB_CSV = "sample_submission.csv"
TRAIN_DIR = os.path.join("train", "train") if os.path.exists(os.path.join("train", "train")) else "train"
TEST_DIR = os.path.join("test", "test") if os.path.exists(os.path.join("test", "test")) else "test"

# 2. Read Labels and Define Explicit Integer Mappings
df = pd.read_csv(TRAIN_CSV)

# Alphabetical sorting ensures strict deterministic mapping across sessions
CLASSES = sorted(df["class"].unique())
CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: cls_name for idx, cls_name in enumerate(CLASSES)}

df["target"] = df["class"].map(CLASS_TO_IDX)

print("=== Target Class Encoding (0 to 7) ===")
for cls_name, idx in CLASS_TO_IDX.items():
    print(f"  {idx}: {cls_name}")

# 3. Stratified 80/20 Split
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["target"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"\nSplit complete:")
print(f"  Training samples  : {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Validation samples: {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")

# Confirm stratified balance across splits
split_check = pd.DataFrame({
    "Train %": (train_df["class"].value_counts(normalize=True) * 100).round(2),
    "Val %": (val_df["class"].value_counts(normalize=True) * 100).round(2)
})
print("\nClass proportions preserved across folds:")
print(split_check)

ModuleNotFoundError: No module named 'torch'